# PhysicsLLM: evaluate Qwen and Llama

Runs open-weight models on the **same three test sets** as the hosted models (117 multiple choice, 25 tutor queries, 10 safety tests), with **retrieval on and off**. It uses the same `rag.py` retrieval, `tutor.py` rules and answer parsing as `run_eval.py`, so the numbers sit in the same table as GPT-5.4 mini and PhysicsSLM.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Have `colab_bundle.zip` from the PhysicsLLM folder ready (you'll upload it in step 2, or put it in `MyDrive/PhysicsLLM/`).
3. For Llama only: accept the license at huggingface.co/meta-llama/Llama-3.2-1B-Instruct (and 3B), then add a Colab secret named `HF_TOKEN` (key icon on the left).

Results are saved to Google Drive after every batch, so if Colab disconnects, just run all cells again: finished answers are skipped.

**Rough time on a T4:** Qwen 0.5B ≈ 15 min · Qwen 1.5B ≈ 30 min · Llama 1B ≈ 25 min · Llama 3B ≈ 60 min (all three sets, RAG on and off).

## 1. Check the GPU and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate

## 2. Project files (Drive keeps the results safe)

In [ ]:
import os, shutil, zipfile, sys
from google.colab import drive, files
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/PhysicsLLM'
OUT = WORK + '/raw'
os.makedirs(OUT, exist_ok=True)
bundle = WORK + '/colab_bundle.zip'
if not os.path.exists(bundle):
    print('Upload colab_bundle.zip from your PhysicsLLM folder:')
    up = files.upload()
    shutil.move(list(up)[0], bundle)
shutil.rmtree('/content/physics', ignore_errors=True)
zipfile.ZipFile(bundle).extractall('/content/physics')
sys.path.insert(0, '/content/physics')
os.chdir('/content/physics')
import colab_eval as CE
print('Ready. Multiple-choice items:', len(CE.items_for('mcq')), '| queries:', len(CE.items_for('queries')), '| safety:', len(CE.items_for('safety')))

## 3. Choose what to run

In [ ]:
# Any of: 'qwen2.5-0.5b', 'qwen2.5-1.5b', 'qwen2.5-3b', 'llama3.2-1b', 'llama3.2-3b'
MODELS_TO_RUN = ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'llama3.2-1b', 'llama3.2-3b']
SETS = ['mcq', 'queries', 'safety']
RAG_MODES = [True, False]      # retrieval on and off, like the hosted models
BATCH = 8                      # answers generated together; lowered automatically if memory runs out
LIMIT = 0                      # e.g. 3 for a quick test; 0 = everything

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
print('Hugging Face token found' if HF_TOKEN else 'No HF_TOKEN secret: Qwen will run, Llama will be skipped')

## 4. Run (safe to re-run after a disconnect)

In [ ]:
import time
for key in MODELS_TO_RUN:
    repo = CE.OPEN_MODELS[key]
    if key.startswith('llama') and not HF_TOKEN:
        print(f'Skipping {key}: needs the HF_TOKEN secret and the accepted Llama license.')
        continue
    print(f'\n=== {key}  ({repo}) ===')
    t0 = time.time()
    try:
        model = CE.HFModel(repo, token=HF_TOKEN)
    except Exception as e:
        print(f'Could not load {repo}: {type(e).__name__}: {e}')
        continue
    try:
        CE.run_model(key, model, sets=SETS, rag_modes=RAG_MODES, out_dir=OUT, batch=BATCH, limit=LIMIT)
    finally:
        model.close()
    print(f'{key} finished in {(time.time() - t0) / 60:.0f} min')

## 5. Quick look at multiple-choice accuracy

In [ ]:
CE.quick_summary(OUT)

## 6. Download the results

This downloads `open_model_results.zip`. On your laptop:
1. Unzip it into `PhysicsLLM\evaluation\raw\` (files like `mcq_qwen2.5-1.5b.jsonl`).
2. In PowerShell:
```
cd C:\Users\14158\PhysicsLLM
.venv\Scripts\python.exe run_eval.py --rebuild --set all
```
That merges Qwen and Llama into the same CSVs, summary and Results page as GPT-5.4 mini and PhysicsSLM, and keeps any scores you already typed in.

In [ ]:
shutil.make_archive('/content/open_model_results', 'zip', OUT)
files.download('/content/open_model_results.zip')